In [ ]:
import os
import uuid
from qdrant_client import QdrantClient, models
from fastembed import TextEmbedding, SparseTextEmbedding, LateInteractionTextEmbedding
from dotenv import load_dotenv
from utils.semantic_chuncker import SemanticChunker
from utils.edgar_client import EdgarClient

load_dotenv()

/Users/luizfelipew/Documents/git/AI-Engineering/dev-eficiente-IA/engineering-ai/curso-ia/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [ ]:
DENSE_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
SPARSE_MODEL = "Qdrant/bm25"
COLBERT_MODEL = "colbert-ir/colbertv2.0"
COLLECTION_NAME = "financial"
EMAIL = os.getenv("EMAIL_EDGAR")
# FILE_PATH = "./AAPL_10-K_1A_temp.md"
MAX_TOKENS = 300

qdrant = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY"),
)


/var/folders/8j/v8rrkr_d34xg4q6h2m2gv6hc0000gn/T/ipykernel_79737/2295309268.py:9: UserWarning: Failed to obtain server version. Unable to check client-server compatibility. Set check_compatibility=False to skip version check.
  qdrant = QdrantClient(


In [ ]:
# from markdown_it.rules_block.paragraph import paragraph
# with open(FILE_PATH, "r", encoding="utf-8") as f:
#     content = f.read()
    
# paragraphs = content.split("\n\n")
# chunks = [p.strip() for p in paragraphs if len(p.strip()) > 50]

edgar = EdgarClient(email=EMAIL)
data_10k = edgar.fetch_filing_data(ticker="AAPL", form_type="10-K")
text_10k = edgar.get_combine_text(data_10k)

data_10q = edgar.fetch_filing_data(ticker="AAPL", form_type="10-Q")
text_10q = edgar.get_combine_text(data_10q)

chunker = SemanticChunker(max_tokens=MAX_TOKENS)
# chunks = chunker.create_chunks(content)

all_chunks = []
for data, text in [(data_10k, text_10k), (data_10q, text_10q)]:
    chunks = chunker.create_chunks(text)
    for chunk in chunks:
        all_chunks.append({"text": chunk, "metadata": data["metadata"]})



In [ ]:
dense_model = TextEmbedding(DENSE_MODEL)
sparse_model = SparseTextEmbedding(SPARSE_MODEL)
colbert_model = LateInteractionTextEmbedding(COLBERT_MODEL)

points = []
for chunk_data in all_chunks:
    chunk = chunk_data["text"]
    metadata = chunk_data["metadata"]
        
    dense_embedding = list(dense_model.passage_embed([chunk]))[0].tolist()
    sparse_embedding = list(sparse_model.passage_embed([chunk]))[0].as_object()
    # ColBERT retorna múltiplos vetores (multivector) - precisa ser uma lista de listas
    colbert_vectors = list(colbert_model.passage_embed([chunk]))[0]
    colbert_embedding = [vec.tolist() for vec in colbert_vectors]
    
    point = models.PointStruct(
        id=str(uuid.uuid4()),
        vector={
            "dense": dense_embedding,
            "sparse": sparse_embedding,
            "colbert": colbert_embedding
        },
        # payload={"text": chunk, "source": FILE_PATH},
        payload={"text": chunk, "metadata": metadata},
    )
    points.append(point)

qdrant.upload_points(collection_name=COLLECTION_NAME, points=points, batch_size=5)


Fetching 5 files: 100%|██████████| 5/5 [00:17<00:00,  3.48s/it]
